# Data Handling & Visualization

## Week 01 : Missing Values, Duplicates & Data Types

In [4]:
import pandas as pd
import numpy as np

salary = pd.read_csv('../../Data/Slot_02/salary_survey_raw.csv')

salary_clean = pd.read_csv('salary_cleaned.csv')

### 1.1 - Missing Values

In [5]:
def missing_report(salary):
    # Tạo báo cáo chi tiết về số lượng và tỷ lệ missing values.
    miss = salary.isnull().sum()
    pct = (miss / len(salary) * 100).round(2)
    
    report = pd.DataFrame({
        'Count Missing': miss, 
        'Percent Missing': pct
    })
    return report.query('`Count Missing` > 0')\
                    .sort_values('Percent Missing', ascending=False)

print(missing_report(salary))

                                 Count Missing  Percent Missing
income_context                            2269            81.04
us_state                                  1813            64.75
city                                      1562            55.79
additional_monetary_comp                  1538            54.93
additional_context_on_job_title            969            34.61
race                                       766            27.36
annual_salary                              391            13.96
country                                    115             4.11
gender                                      69             2.46


#### Assess

GROUP 01: Missing Not A Random (MNAR)
- additional_monetary_comp : The field is usually left blank because they don't have any bonus/allowance income. -> Fill: 0
- us_state : This position may be vacant because the candidate works in a country other than the United States. -> Fill: Non-US if they don't have a US state, otherwise fill Unknown.
- income_context : The defect rate exceeds the information security threshold (80%) -> Use feature droping 
- city : The missing rate is high (>50%). This is likely conditional missingness (MNAR) where respondents deliberately chose not to disclose granular location data for privacy reasons. -> Action: Fill with 'Unknown'.

GROUP 02: Missing At Random (MAR)
- additional_context_on_job_title : Identifying sensitive information, the absence of such information is subject to the subject's right to refuse to provide it. -> Fill: Not Disclosed 
- race : Open-ended question, not mandatory. "Missing" means no additional information provided -> Fill: No additional Context.

GROUP 03: Target Variable / Dependent Variable
- annual_salary : In scientific research methodology, imputing missing values in the target variable may introduce synthetic data bias, thereby reducing the statistical reliability of tests such as ANOVA or T-test. -> Action: Sort the dataset by 'industry', 'job_title', and 'years_of_experience_overall' first. Then apply `.interpolate(method='linear')` grouped by these categories.

GROUP 04: Missing Completely At Random (MCAR)
- country : The missing rate is extremely low (< 5%) and represents completely random noise. Using mode imputation preserves the sample size without distorting the overall probability distribution. -> Fill: Mode
- gender : The missing rate is statistically negligible. Imputing with the most frequent value is the optimal standard approach to maintain data integrity for categorical variables. -> Fill: Mode


#### Cleaned Dataset

In [7]:
import pandas as pd 
import numpy as np

df = pd.read_csv('../../Data/Slot_02/salary_survey_raw.csv')
df.head()

def clean_currency(x):
    if pd.isna(x): return np.nan
    if isinstance(x, str):
        try: 
            return float(x.replace('$', '').replace(',', '').strip())
        except ValueError: 
            return np.nan
    return float(x)

df['annual_salary'] = df['annual_salary'].apply(clean_currency)
df['additional_monetary_comp'] = df['additional_monetary_comp'].apply(clean_currency)

# GROUP 01: MNAR
df['additional_monetary_comp'] = df['additional_monetary_comp'].fillna(0)

us_aliases = ['united states', 'us', 'usa', 'united states of america']
is_not_us = ~df['country'].str.strip().str.lower().isin(us_aliases) & df['country'].notna()
df.loc[is_not_us & df['us_state'].isna(), 'us_state'] = 'Non-US'
df['us_state'] = df['us_state'].fillna('Unknown')
df['city'] = df['city'].fillna('Unknown')

df = df.drop(columns=['income_context'])

# GROUP 02: MAR
df['additional_context_on_job_title'] = df['additional_context_on_job_title'].fillna('Not Disclosed')
df['race'] = df['race'].fillna('No additional Context.')

# GROUP 03: Target Variable
df = df.sort_values(by=['industry', 'job_title', 'years_of_experience_overall'])
df['annual_salary'] = df.groupby(['industry', 'job_title', 'years_of_experience_overall'])['annual_salary'].transform(lambda x: x.interpolate(method='linear'))
df = df.dropna(subset=['annual_salary'])

# GROUP 04: MCAR
df['country'] = df['country'].fillna(df['country'].mode()[0])
df['gender'] = df['gender'].fillna(df['gender'].mode()[0])

# 3. EXPORT TO EXCEL/CSV
df.to_csv('salary_cleaned.csv', index=False)
print("Pipeline hoàn tất. Đã xuất file: salary_cleaned.csv")

Pipeline hoàn tất. Đã xuất file: salary_cleaned.csv


#### Audit & Comparison

In [8]:
import pandas as pd

df_raw = pd.read_csv('../../Data/Slot_02/salary_survey_raw.csv')
df_clean = pd.read_csv('salary_cleaned.csv')

# Check Feature
def audit_missing_data(df_before, df_after):
    all_cols = df_before.columns
    null_before = df_before.isnull().sum()
    null_after = pd.Series(index=all_cols, dtype=object)
    
    for col in all_cols:
        if col in df_after.columns:
            null_after[col] = df_after[col].isnull().sum()
        else:
            null_after[col] = 'Dropped'

    audit_df = pd.DataFrame({
        'Null_Before': null_before,
        'Null_After': null_after
    })
    
    audit_df['Status'] = audit_df['Null_After'].apply(
        lambda x : 'Cleaned' if x == 0 else ('Removed' if x == 'Dropped' else f'Remaining: {x}')
    )
    
    print("="*55)
    print("DATASET QUALITY AUDIT REPORT")
    print(f"Original Shape : {df_before.shape}")
    print(f"Cleaned Shape  : {df_after.shape}")
    print(f"Rows dropped   : {df_before.shape[0] - df_after.shape[0]} (Listwise Deletion)")
    print(f"Cols dropped   : {df_before.shape[1] - df_after.shape[1]} (Feature Removal)")
    print("="*55)
    
    return audit_df

audit_result = audit_missing_data(df_raw, df_clean)
audit_result

DATASET QUALITY AUDIT REPORT
Original Shape : (2800, 17)
Cleaned Shape  : (2641, 16)
Rows dropped   : 159 (Listwise Deletion)
Cols dropped   : 1 (Feature Removal)


,Null_Before,Null_After,Status
timestamp,0,0,Cleaned
how_old_are_you,0,0,Cleaned
industry,0,0,Cleaned
job_title,0,0,Cleaned
additional_context_on_job_title,969,0,Cleaned
annual_salary,391,0,Cleaned
additional_monetary_comp,1538,0,Cleaned
currency,0,0,Cleaned
income_context,2269,Dropped,Removed
country,115,0,Cleaned


### 1.2 - Duplicates

#### Check and retrieve duplicate data.

In [5]:
# Indetify Core Identifying Characteristics
core_features = ['how_old_are_you', 'industry', 'job_title', 'annual_salary', 'gender', 'city']

# Statistic Number Duplicates
exact_dupli = salary_clean.duplicated().sum()
partial_dupli = salary_clean.duplicated(subset=core_features).sum()

print("BÁO CÁO KHẢO SÁT TRÙNG LẶP:")
print(f"- Exact Duplicates (Trùng 100% - Lỗi hệ thống): {exact_dupli} dòng")
print(f"- Partial Duplicates (Trùng cục bộ - Nghi ngờ Resubmit): {partial_dupli} dòng\n")

# 2. Truy xuất các dòng trùng lặp để quan sát (Inspection)
# keep=False: Giữ lại tất cả các phiên bản của dòng trùng lặp để đối chiếu
salary_dupli = salary_clean[salary_clean.duplicated(subset=core_features, keep=False)]

# Sort theo core_features và timestamp để các cặp lặp đứng liền kề nhau
salary_dupli_sorted = salary_dupli.sort_values(by=core_features + ['timestamp'])

print("DANH SÁCH CÁC DÒNG TRÙNG LẶP (HIỂN THỊ 10 DÒNG ĐẦU):")
display(salary_dupli_sorted.head(10))

BÁO CÁO KHẢO SÁT TRÙNG LẶP:
- Exact Duplicates (Trùng 100% - Lỗi hệ thống): 36 dòng
- Partial Duplicates (Trùng cục bộ - Nghi ngờ Resubmit): 74 dòng

DANH SÁCH CÁC DÒNG TRÙNG LẶP (HIỂN THỊ 10 DÒNG ĐẦU):


,timestamp,how_old_are_you,industry,job_title,additional_context_on_job_title,annual_salary,additional_monetary_comp,currency,country,us_state,city,years_of_experience_in_field,years_of_experience_overall,highest_level_of_education,gender,race
502,04/17/2021,18-24,Education (Higher Ed),Postdoc,Not Disclosed,94879.0,0.0,EUR,United States,Colorado,Unknown,8 - 10 years,8 - 10 years,Some college,Woman,Black or African American
503,2021-04-15 06:29,18-24,Education (Higher Ed),Postdoc,Not Disclosed,94879.0,0.0,EUR,United States,Colorado,Unknown,8 - 10 years,8 - 10 years,Some college,Woman,Black or African American
1160,04/22/2021,18-24,HR/People Ops,Chief People Officer,Not Disclosed,73949.0,0.0,INR,United States,Unknown,Unknown,8 - 10 years,8 - 10 years,Some college,Male,White
1159,04/28/2021 00:19:45,18-24,HR/People Ops,Chief People Officer,Not Disclosed,73949.0,0.0,INR,United States,Unknown,Unknown,8 - 10 years,8 - 10 years,Some college,Male,White
1179,04/29/2021 18:58:08,18-24,HR/People Ops,HR Business Partner,Government pay scale,41478.0,2988.0,USD,Germany,Non-US,Unknown,5-7 years,5-7 years,College degree,Man,White
1175,4/20/2021 11:52:00,18-24,HR/People Ops,HR Business Partner,Government pay scale,41478.0,2988.0,USD,Germany,Non-US,Unknown,5-7 years,5-7 years,College degree,Man,White
1340,4/24/2021 13:43:50,18-24,Healthcare,Medical Director,Contract role,175686.0,11451.0,GBP,Canada,Non-US,Unknown,8 - 10 years,8 - 10 years,Some college,Woman,Asian or Asian American
1342,4/24/2021 13:43:50,18-24,Healthcare,Medical Director,Contract role,175686.0,11451.0,GBP,Canada,Non-US,Unknown,8 - 10 years,8 - 10 years,Some college,Woman,Asian or Asian American
1335,4/13/2021 6:21:16,18-24,Healthcare,Medical Director,Not Disclosed,294966.0,0.0,USD,Australia,Non-US,Unknown,21 - 30 years,21 - 30 years,"Professional degree (MD, JD, etc.)",Male,"Hispanic, Latino, or Spanish origin"
1337,4/13/2021 6:21:16,18-24,Healthcare,Medical Director,Not Disclosed,294966.0,0.0,USD,Australia,Non-US,Unknown,21 - 30 years,21 - 30 years,"Professional degree (MD, JD, etc.)",Male,"Hispanic, Latino, or Spanish origin"


#### Execute Removal

In [6]:
# Lưu cấu trúc trước khi xóa
shape_before = salary_clean.shape

# Thực thi loại bỏ - Giữ lại bản ghi có timestamp muộn nhất (keep='last') 
salary_clean = salary_clean.drop_duplicates(subset=core_features, keep='last')

# 3. Đối chiếu Audit
print("="*50)
print("AUDIT XỬ LÝ TRÙNG LẶP")
print(f"Shape ban đầu  : {shape_before}")
print(f"Shape sau xóa  : {salary_clean.shape}")
print(f"Số dòng đã drop: {shape_before[0] - salary_clean.shape[0]}")
print("="*50)

AUDIT XỬ LÝ TRÙNG LẶP
Shape ban đầu  : (2641, 16)
Shape sau xóa  : (2567, 16)
Số dòng đã drop: 74


#### Explain

1. Phân tích nguyên nhân (Root Cause Analysis):
Sự trùng lặp xuất phát từ hai nguyên nhân chính:
- Trùng lặp toàn hàng (Exact Duplicates): Khớp 100% dữ liệu bao gồm cả `timestamp`. Đây là lỗi hệ thống lưu trữ (System/Network Glitch), xảy ra khi có độ trễ mạng khiến một truy vấn nộp form bị nhân bản.
- Trùng lặp cục bộ theo Key Columns (Partial Duplicates): Trùng khớp các đặc trưng nhân khẩu học và công việc nhưng lệch `timestamp`. Đây là hành vi cập nhật dữ liệu (Resubmission), xảy ra khi đối tượng thực nghiệm phát hiện sai sót và nộp lại form để đính chính.
- Chiến lược: Áp dụng `.drop_duplicates()` với tham số `keep='last'` nhằm loại bỏ nhiễu hệ thống, đồng thời ưu tiên giữ lại bản cập nhật thông tin muộn nhất và chính xác nhất từ người dùng.

2. Báo cáo kiểm định cấu trúc (Shape Audit):
- Số dòng trước khi xử lý: 2641
- Số dòng sau khi xử lý: 2567
- Tổng số dòng nhiễu đã loại bỏ: 74